# 3. CPTAC validation results

Consolidated replacement for `3. results_tmt.ipynb`, `3. results_labelfree.ipynb`, and `3. results_itraq.ipynb`.

This notebook checks the saved best-model result files, evaluates the same coefficient-dictionary models on TMT, LabelFree, and iTRAQ validation data, compares all tissues versus a filtered-tissue view, and produces compact summary plots.

In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "3.results_cptac_loo"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import src.utils_val as uv
from src.utils_val import (
    load_best_result_tables,
    load_batch_result_tables,
    compare_model_result_tables,
    evaluate_cptac_validation_datasets,
    summarize_cptac_results,
    plot_cptac_summary,
    plot_cptac_metric_distributions,
    plot_cptac_tissue_mse,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 180, "axes.titleweight": "bold"})

## 1. Configuration

Paths match the original notebooks. `results_val` holds the trained coefficient models; `val_data` holds normalized validation matrices generated by notebook 2.

In [ ]:
RESULTS_DIR = Path("../data/loo/results_zscore")
VAL_DATA_DIR = Path("../data/val_data")
OUTPUT_DIR = Path("../data/loo/results_zscore/plots_cptac_unified")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATASET_CONFIGS = {
    "TMT": {
        "rna_path": VAL_DATA_DIR / "TMT_transcriptomics.joblib",
        "prot_path": VAL_DATA_DIR / "TMT_proteomics.joblib",
    },
    "LabelFree": {
        "rna_path": VAL_DATA_DIR / "LabelFree_transcriptomics.joblib",
        "prot_path": VAL_DATA_DIR / "LabelFree_proteomics.joblib",
    },
    "iTRAQ": {
        "rna_path": VAL_DATA_DIR / "iTRAQ_transcriptomics.joblib",
        "prot_path": VAL_DATA_DIR / "iTRAQ_proteomics.joblib",
    },
}

# Same filtered-tissue list used in the original split notebooks.
EXCLUDED_TISSUES = ["RENAL_2019", "HEADNECK_2021", "PANCREA2021"]

# Original notebook skipped models with missing RNA features. Keep that behavior here.
IMPUTE_MISSING_FEATURES = False

print("Results directory:", RESULTS_DIR.resolve())
print("Validation data directory:", VAL_DATA_DIR.resolve())

## 2. Best-model and batch-file consistency check

In [ ]:
best_models, best_files = load_best_result_tables(RESULTS_DIR)
batch_models, batch_files = load_batch_result_tables(RESULTS_DIR)
comparison = compare_model_result_tables(best_models, batch_models)

print(f"Best-model files: {len(best_files)}")
print(f"Batch files:      {len(batch_files)}")
print(f"Best rows:        {len(best_models):,}")
print(f"Batch rows:       {len(batch_models):,}")
display(pd.DataFrame([comparison]).drop(columns=["missing_in_batch", "missing_in_best"], errors="ignore"))

if comparison.get("missing_in_batch"):
    print("Proteins in best files but not batch files:", comparison["missing_in_batch"][:10])
if comparison.get("missing_in_best"):
    print("Proteins in batch files but not best files:", comparison["missing_in_best"][:10])

display(best_models.head())

## 3. Evaluate all CPTAC validation datasets

Each dataset is evaluated twice: first with all tissues, then after dropping the three known high-error tissue groups used in the original notebooks.

In [ ]:
cptac_results = evaluate_cptac_validation_datasets(
    DATASET_CONFIGS,
    best_models,
    excluded_tissues=EXCLUDED_TISSUES,
    impute_missing_features=IMPUTE_MISSING_FEATURES,
)

summary_df = summarize_cptac_results(cptac_results)
print("Summary rows:", len(summary_df))
display(summary_df.round(4))

## 4. Feature coverage and zero inflation checks

In [ ]:
coverage_rows = []
zero_rows = []
for (dataset, scope), bundle in cptac_results.items():
    cov = bundle["coverage_df"].copy()
    if len(cov):
        cov["Dataset"] = dataset
        cov["Scope"] = scope
        coverage_rows.append(cov)
    zeros = bundle["zero_stats"].copy()
    if len(zeros):
        zeros["Dataset"] = dataset
        zeros["Scope"] = scope
        zero_rows.append(zeros)

coverage_df = pd.concat(coverage_rows, ignore_index=True) if coverage_rows else pd.DataFrame()
zero_stats_df = pd.concat(zero_rows, ignore_index=True) if zero_rows else pd.DataFrame()

if len(coverage_df):
    coverage_summary = (
        coverage_df.groupby(["Dataset", "Scope"], as_index=False)
        .agg(
            models=("Protein", "nunique"),
            median_missing_fraction=("missing_fraction", "median"),
            max_missing_fraction=("missing_fraction", "max"),
        )
    )
    display(coverage_summary.round(4))

if len(zero_stats_df):
    print("Highest RNA zero percentages by dataset/scope")
    display(
        zero_stats_df.sort_values("percent_zeros", ascending=False)
        .groupby(["Dataset", "Scope"], as_index=False)
        .head(5)
        .round(3)
    )

## 5. Compact performance plots

In [ ]:
fig = plot_cptac_summary(summary_df)
if fig is not None:
    fig.savefig(OUTPUT_DIR / "cptac_summary.png", bbox_inches="tight")

In [ ]:
long_results = []
for (dataset, scope), bundle in cptac_results.items():
    df = bundle["results_df"].copy()
    if len(df):
        df["Dataset"] = dataset
        df["Scope"] = scope
        long_results.append(df)

long_results_df = pd.concat(long_results, ignore_index=True) if long_results else pd.DataFrame()
plot_cptac_metric_distributions(long_results_df)

In [ ]:
tissue_rows = []
for (dataset, scope), bundle in cptac_results.items():
    tdf = bundle["tissue_df"].copy()
    if len(tdf):
        tissue_rows.append(tdf)

tissue_mse_df = pd.concat(tissue_rows, ignore_index=True) if tissue_rows else pd.DataFrame()
plot_cptac_tissue_mse(tissue_mse_df)

## 6. Top and bottom proteins

A concise table replaces the repeated per-dataset top-list cells from the original notebooks.

In [ ]:
if len(long_results_df):
    top_corr = (
        long_results_df.dropna(subset=["Correlation"])
        .sort_values(["Dataset", "Scope", "Correlation"], ascending=[True, True, False])
        .groupby(["Dataset", "Scope"], as_index=False)
        .head(10)
    )
    best_mse = (
        long_results_df.dropna(subset=["MSE"])
        .sort_values(["Dataset", "Scope", "MSE"], ascending=[True, True, True])
        .groupby(["Dataset", "Scope"], as_index=False)
        .head(10)
    )
    print("Top 10 proteins by correlation per dataset/scope")
    display(top_corr[["Dataset", "Scope", "Protein", "Correlation", "MSE", "Cosine_Similarity"]].round(4))
    print("Top 10 proteins by lowest MSE per dataset/scope")
    display(best_mse[["Dataset", "Scope", "Protein", "MSE", "Correlation", "Cosine_Similarity"]].round(4))

## 7. Optional exports

In [ ]:
SAVE_TABLES = False
if SAVE_TABLES:
    summary_df.to_csv(OUTPUT_DIR / "cptac_summary.csv", index=False)
    long_results_df.to_csv(OUTPUT_DIR / "cptac_protein_results.csv", index=False)
    tissue_mse_df.to_csv(OUTPUT_DIR / "cptac_tissue_mse.csv", index=False)
    coverage_df.to_csv(OUTPUT_DIR / "cptac_feature_coverage.csv", index=False)
    print("Saved tables to", OUTPUT_DIR.resolve())